# Storage Menager

Esta é uma documentação detalhada e didática para o módulo de integração com o **Supabase Storage**. O código foi projetado seguindo princípios de Programação Orientada a Objetos (POO) para facilitar a manutenção e o reuso de conexões.

---

## 1. Visão Geral

O módulo é composto por duas classes principais que trabalham em conjunto para abstrair a complexidade da API do Supabase. A ideia central é separar a **configuração da conexão** (infraestrutura) da **manipulação de arquivos** (lógica de negócio).

- **SupabaseConnection**: Garante que a conexão com o servidor seja estabelecida corretamente usando variáveis de ambiente.
- **StorageManager**: Oferece uma interface simplificada para enviar, deletar e recuperar links de arquivos dentro de um "bucket" (balde/repositório) específico.

---

## 2. Fluxo de Execução

Para entender como as classes interagem, imagine o seguinte fluxo:

1. **Carregamento de Ambiente**: O script lê o arquivo `.env` para capturar a URL e a Chave do projeto.
2. **Instanciação**: Ao criar um `StorageManager`, ele verifica se você já possui uma conexão. Se não, ele chama a `SupabaseConnection` automaticamente.
3. **Acesso ao Bucket**: O cliente se conecta ao bucket específico informado no início.
4. **Operação**: Você chama métodos simples (como `upload_bytes`), e a classe lida com a lógica interna da biblioteca oficial do Supabase.

---

## 3. Resumo de Métodos

| **Método** | **Classe** | **Descrição Breve** |
| --- | --- | --- |
| `__init__` | `SupabaseConnection` | Valida credenciais e cria o cliente `Client`. |
| `__init__` | `StorageManager` | Define o bucket alvo e garante a existência de um cliente ativo. |
| `upload_bytes` | `StorageManager` | Faz o upload de arquivos binários (bytes) para o storage. |
| `delete_files` | `StorageManager` | Remove uma lista de arquivos de uma só vez. |
| `get_url` | `StorageManager` | Gera o link público para visualizar o arquivo. |

---

## 4. Arquitetura e Insights

- **Singleton Implícito vs Injeção de Dependência**: A classe `StorageManager` aceita um `supabase_client` opcional. Isso é excelente para testes unitários ou para reaproveitar uma conexão já aberta em outra parte do sistema, evitando múltiplas conexões desnecessárias.
- **Tratamento de Erros**: O uso de blocos `try/except` nos métodos de escrita (upload/delete) evita que a aplicação trave caso ocorra uma falha de rede ou permissão.
- **Segurança**: O uso de `dotenv` e `os.getenv` garante que chaves sensíveis nunca fiquem expostas diretamente no código-fonte.

---

## 5. Documentação Detalhada

### Classe SupabaseConnection

**Descrição**

Responsável por centralizar a autenticação. Ela serve como uma "fábrica" de conexões, garantindo que o cliente do Supabase esteja devidamente configurado antes de qualquer operação.

**Argumentos**

- Não possui argumentos diretos (utiliza variáveis de ambiente `.env`).

**Métodos**

### 1. `__init__`

- **Descrição**: Localiza as variáveis `SUPABASE_URL` e `SUPABASE_SECRET_KEY`, validando-as antes de instanciar o cliente oficial.
- **Argumentos**: Nenhum.
- **Retornos**: Nenhum (inicializa o atributo `self.client`).
- **Raises**: `ValueError` se as chaves não forem encontradas no ambiente.
- **Exemplos**:Python
    
    # 
    
    `conn = SupabaseConnection()
    client = conn.client # Cliente pronto para uso`
    

---

### Classe StorageManager

**Descrição**

Encapsula a lógica de manipulação de buckets. É a classe "front-end" que o desenvolvedor usará na maior parte do tempo para gerenciar arquivos.

**Argumentos**

- `bucket_name` (str): O nome da pasta/repositório no Supabase.
- `supabase_client` (Optional[Client]): Uma instância existente do cliente Supabase.

**Métodos**

### 1. `upload_bytes`

- **Descrição**: Envia um arquivo para o storage a partir de dados em memória (bytes).
- **Argumentos**:
    - `path_on_storage` (str): Caminho/nome do arquivo no destino.
    - `file_bytes` (bytes): O conteúdo binário do arquivo.
    - `content_type` (str): Tipo de arquivo (ex: `image/png`).
- **Retornos**: `dict` com metadados do upload ou `None` em caso de erro.
- **Raises**: Captura exceções genéricas e as imprime no console.
- **Exemplos**:Python
    
    # 
    
    `manager.upload_bytes("docs/perfil.jpg", foto_bytes, "image/jpeg")`
    

### 2. `delete_files`

- **Descrição**: Exclui um ou mais arquivos simultaneamente.
- **Argumentos**:
    - `paths` (List[str]): Lista de caminhos (ex: `["img1.jpg", "img2.jpg"]`).
- **Retornos**: `dict` com o status da remoção ou `None`.
- **Raises**: Captura exceções de rede ou permissão.
- **Exemplos**:Python
    
    # 
    
    `manager.delete_files(["velho/foto1.png", "velho/foto2.png"])`
    

### 3. `get_url`

- **Descrição**: Obtém o endereço web público para acessar um arquivo.
- **Argumentos**:
    - `path` (str): Caminho do arquivo dentro do bucket.
- **Retornos**: `str` contendo a URL completa.
- **Raises**: `ValueError` se o caminho estiver vazio.
- **Exemplos**:Python
    
    # 
    
    `link = manager.get_url("produtos/celular.png")
    print(link) # https://xyz.supabase.co/storage/v1/object/public/...`
    
